# Fase 2 — Limpieza e ingeniería de datos

| Campo | Valor |
|---|---|
| **Entrada** | `data/raw/chembl_panama_bioactivity.csv` |
| **Salidas** | `activities_clean.csv` (medición) + `compounds_features.csv` (107 compuestos) |
| **Doc** | [`docs/analisis_proyecto/fases/fase2_limpieza_datos.md`](../docs/fases/fase2_limpieza_datos.md) |

Unidad principal de análisis: **compuesto** (107 filas). La tabla de mediciones conserva `standard_relation`, `target_chembl_id` y `standard_type` para perfiles de dianas/endpoints.


## 0. Configuración

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

from src.paths import PROJECT_ROOT as ROOT, setup_path
setup_path()

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams.update({"figure.figsize": (10, 5), "figure.dpi": 120})

FIG_DIR = ROOT / "outputs" / "chembl" / "figures"
RESULTS_DIR = ROOT / "outputs" / "chembl" / "results"
for d in (FIG_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

from src.analisis_proyecto.chembl_preprocessing import (
    FEATURE_COLS,
    build_compound_features,
    drop_columns_high_nan,
    filter_potential_duplicates,
    impute_median_by_family,
    load_bioactivity,
    missingness_upset_series,
    numeric_and_categorical_cols,
    pchembl_imputation_report,
    plot_missingno_report,
)

RAW_CSV = ROOT / "data" / "raw" / "chembl_panama_bioactivity.csv"
ACTIVITIES_CSV = ROOT / "data" / "processed" / "activities_clean.csv"
COMPOUNDS_CSV = ROOT / "data" / "processed" / "compounds_features.csv"
NAN_COL_THRESHOLD = 250

assert RAW_CSV.exists(), f"No se encontró {RAW_CSV}. Ejecuta fase1_adquisicion.ipynb"

raw = load_bioactivity(RAW_CSV)
print(f"RAW: {raw.shape[0]:,} filas × {raw.shape[1]} columnas | compuestos: {raw['chembl_id'].nunique()}")
if "potential_duplicate" in raw.columns:
    n_dup = int((raw["potential_duplicate"].fillna(0).astype(int) == 1).sum())
    print(f"Filas potential_duplicate=1 (a eliminar): {n_dup}")


## 1. Diagnóstico de faltantes

Visualización sobre datos RAW (antes de dedup). Se conserva para documentar el estado inicial.


In [ ]:
saved_msno = plot_missingno_report(raw, FIG_DIR)
if saved_msno:
    for path in saved_msno:
        display(Image(filename=str(path)))

upset_data, nan_cols = missingness_upset_series(raw)
if upset_data is not None:
    from upsetplot import UpSet
    upset = UpSet(upset_data, subset_size="count", show_counts=True)
    upset.plot()
    plt.suptitle("Patrones de valores faltantes (UpSet)")
    plt.savefig(FIG_DIR / "missingness_upset.png", bbox_inches="tight")
    plt.show()


## 2. Dedup + imputación + tablas de salida

1. Eliminar `potential_duplicate==1` (corrige ~801 filas duplicadas).
2. Drop columnas con >250 NaN e imputación por mediana de `family`.
3. Guardar `activities_clean.csv` (nivel medición).
4. Agregar `compounds_features.csv` (107 compuestos).


In [ ]:
activities, dup_report = filter_potential_duplicates(raw)
display(dup_report)

activities, nan_report = drop_columns_high_nan(activities, threshold=NAN_COL_THRESHOLD)
num_cols, cat_cols = numeric_and_categorical_cols(activities)
activities = impute_median_by_family(activities, numeric_cols=num_cols, categorical_cols=cat_cols)

ACTIVITIES_CSV.parent.mkdir(parents=True, exist_ok=True)
activities.to_csv(ACTIVITIES_CSV, index=False)

compounds = build_compound_features(activities)
assert compounds["chembl_id"].nunique() == len(compounds), "Debe haber 1 fila por compuesto"
compounds.to_csv(COMPOUNDS_CSV, index=False)

print(f"activities: {activities.shape} | compounds: {compounds.shape}")
print(pchembl_imputation_report(activities))

if "standard_relation" in activities.columns:
    rel = activities["standard_relation"].value_counts(dropna=False)
    print("\nCensura / relación standard_relation (conservada en activities_clean):")
    display(rel.to_frame("n"))


---
*Anterior:* [`fase1_adquisicion.ipynb`](fase1_adquisicion.ipynb)  
*Siguiente:* [`fase3_eda.ipynb`](fase3_eda.ipynb)
